In [1]:
import pandas as pd

In [2]:
# Load your preprocessed tweet data
tweets_df = pd.read_csv('cleaned-sentiments.csv')

In [3]:
# Define a dictionary to map specific words to emojis
emoji_dict = {
    'win': '🏆',
    'gold': '🥇',
    'medal': '🏅',
    'champion': '👑',
    'congratulate': '🎉',
    'team': '👥',
    'game': '🎮',
    'world': '🌎',
    'record': '📈'
}

In [4]:
# Function to replace words in text with corresponding emojis
def text_to_emoji(text, emoji_dict):
    words = text.strip("[]").replace("'", "").split(", ")
    converted_text = ' '.join(emoji_dict.get(word, word) for word in words)
    return converted_text

In [5]:
print(tweets_df.columns)

Index(['Category', 'Keyword', 'Tweet_Timestamp', 'Tweet_Content',
       'Tweet_Number_of_Retweets', 'Tweet_Number_of_Likes',
       'Tweet_Number_of_Looks', 'Lemmatized_Tweets', 'Clean_Tweets'],
      dtype='object')


In [6]:
# Apply the function to each tweet
tweets_df['Cleaned_Tweet_Emojified'] = tweets_df['Clean_Tweets'].apply(lambda x: text_to_emoji(x, emoji_dict))

In [7]:
# Save or display the DataFrame with the new emoji column
print(tweets_df[['Clean_Tweets', 'Cleaned_Tweet_Emojified']].head())

                                        Clean_Tweets  \
0  ['olympic', 'legend', 'usabmnt', 'paris2024', ...   
1  ['olympicrecord', 'rizki', 'juniansyah', 'iwfn...   
2  ['men', 'javelin', 'throw', 'final', 'big', 'a...   
3  ['time', 'alive', 'usabmnt', 'paris2024', 'bas...   
4  ['quincy', 'hall', 'effort', 'stretch', 'win',...   

                             Cleaned_Tweet_Emojified  
0        olympic legend usabmnt paris2024 basketball  
1  olympicrecord rizki juniansyah iwfnet weightli...  
2  men javelin throw final big arshad nadeem neer...  
3            time alive usabmnt paris2024 basketball  
4  quincy hall effort stretch 🏆 olympic 🥇 deserve...  


In [8]:
# Define hate and non-hate keywords and emojis
hate_keywords = {
    'hate', 'anger', 'disgust', 'shame', 'mock', 'ruin', 'disrespect', 'dishonor',
    'fail', 'lose', 'disappoint', 'weak', 'cheat', 'unfair', 'corrupt', 'scam', 
    'boycott', 'ban', 'protest', 'controversy', 'biased', 'bad', 'terrible', 
    'awful', 'useless', 'overrated', 'waste', 'flop', 'loser', 'choker', 'fraud', 
    'pathetic', 'expensive', 'pollution', 'chaos'
}

In [9]:
non_hate_keywords = {'win', 'congrats', 'love', 'happy', 'excite', 'enjoy', 'amazing', 'proud'}

In [10]:
hate_emojis = {'😡', '😠', '👎', '💀'}

In [11]:
non_hate_emojis = {'😊', '🎉', '👍', '❤️', '🏆', '🥇'}

In [12]:
# Function to label tweets as hate or non-hate based on both text and emoji
def label_tweet(text):
    # Convert text to lowercase for consistent matching
    text = text.lower()
    
    # Check for any hate keywords or emojis
    hate_present = any(keyword in text for keyword in hate_keywords) or any(emoji in text for emoji in hate_emojis)
    non_hate_present = any(keyword in text for keyword in non_hate_keywords) or any(emoji in text for emoji in non_hate_emojis)
    
    # Label based on presence of hate indicators
    if hate_present:
        return 'hate'
    elif non_hate_present:
        return 'non-hate'
    else:
        # Default to non-hate if no strong indicators
        return 'non-hate'

In [13]:
# Apply the labeling function to the 'Cleaned_Tweet_Emojified' column in tweets_df
tweets_df['Label'] = tweets_df['Cleaned_Tweet_Emojified'].apply(label_tweet)

In [14]:
# Display the first few rows to verify
tweets_df[['Cleaned_Tweet_Emojified', 'Label']].head()

,Cleaned_Tweet_Emojified,Label
0,olympic legend usabmnt paris2024 basketball,non-hate
1,olympicrecord rizki juniansyah iwfnet weightli...,non-hate
2,men javelin throw final big arshad nadeem neer...,non-hate
3,time alive usabmnt paris2024 basketball,non-hate
4,quincy hall effort stretch 🏆 olympic 🥇 deserve...,non-hate


In [15]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

In [24]:
# Define the text column to be used for N-gram feature extraction
text_column = 'Cleaned_Tweet_Emojified'

In [25]:
# Function to generate N-gram features
def generate_ngram_features(data, text_column, ngram_range=(1, 3), vectorizer_type='count'):
    """
    Generate N-gram features from text data using CountVectorizer or TfidfVectorizer.

    Args:
    - data: DataFrame containing text data.
    - text_column: Column name with text data.
    - ngram_range: Tuple specifying the range of N-grams (default is (1, 3)).
    - vectorizer_type: 'count' for CountVectorizer, 'tfidf' for TfidfVectorizer.

    Returns:
    - ngram_features: Feature matrix with N-gram features.
    - vectorizer: Fitted vectorizer object.
    """
    if vectorizer_type == 'count':
        vectorizer = CountVectorizer(ngram_range=ngram_range, token_pattern=r'(?u)\b\w+\b|[\U00010000-\U0010FFFF]')
    elif vectorizer_type == 'tfidf':
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, token_pattern=r'(?u)\b\w+\b|[\U00010000-\U0010FFFF]')
    else:
        raise ValueError("Invalid vectorizer_type. Use 'count' or 'tfidf'.")

    # Fit and transform the text data
    ngram_features = vectorizer.fit_transform(data[text_column])

    return ngram_features, vectorizer

In [28]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [29]:
# Generate N-gram features using the Cleaned_Tweet_Emojified column
ngram_range = (1, 3)  # Unigram, Bigram, and Trigram
vectorizer_type = 'tfidf'  # Use 'count' for CountVectorizer
ngram_features, vectorizer = generate_ngram_features(tweets_df, text_column, ngram_range, vectorizer_type)

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [23]:
# Fit and transform the 'Cleaned_Tweet_Emojified' column
ngram_features = vectorizer.fit_transform(tweets_df['Cleaned_Tweet_Emojified'])

C:\Users\Janani\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


ValueError: np.nan is an invalid document, expected byte or unicode string.